# Exercise 1 — Transportation Problem


## Task

Consider the following transportation problem. Find the **cost-minimizing shipping schedule** for cases shipped from two canneries to three warehouses.

- Demand at each warehouse must be met.
- Supply at each cannery must not be exceeded.
- Shipping costs depend on the origin-destination pair.

### Shipping costs (dollars per case)

| Cannery | New York (A) | Chicago (B) | Kansas City (C) |
|---|---:|---:|---:|
| Seattle (I) | 2.5 | 1.7 | 1.8 |
| San Diego (II) | 2.5 | 1.8 | 1.4 |

### Supply

| Cannery | Cases available |
|---|---:|
| Seattle (I) | 350 |
| San Diego (II) | 650 |
| **Total** | **1000** |

### Demand

| Warehouse | Cases demanded |
|---|---:|
| New York (A) | 300 |
| Chicago (B) | 300 |
| Kansas City (C) | 300 |
| **Total** | **900** |

Implement the following model in JuMP:

$$
\min_{x_{cw}} \quad \sum_{c\in\mathcal C}\sum_{w\in\mathcal W} t_{cw}x_{cw}
$$

subject to

$$
\sum_{w\in\mathcal W}x_{cw}\le s_c \qquad \forall c\in\mathcal C,
$$

$$
\sum_{c\in\mathcal C}x_{cw}\ge d_w \qquad \forall w\in\mathcal W,
$$

$$
x_{cw}\ge0 \qquad \forall c\in\mathcal C,\;w\in\mathcal W.
$$

Here, $\mathcal C$ is the set of canneries, $\mathcal W$ the set of warehouses, $t_{cw}$ the transportation cost, $s_c$ supply, $d_w$ demand, and $x_{cw}$ the shipped quantity.


## Reference implementation


In [1]:
using JuMP, HiGHS

C = ["Seattle", "San Diego"]
W = ["New York", "Chicago", "Kansas City"]

ship_mat = [
    2.5 1.7 1.8;
    2.5 1.8 1.4
]

supply_vec = [350, 650]
demand_vec = [300, 300, 300]

t = JuMP.Containers.DenseAxisArray(ship_mat, C, W)
s = JuMP.Containers.DenseAxisArray(supply_vec, C)
d = JuMP.Containers.DenseAxisArray(demand_vec, W)

model = Model(HiGHS.Optimizer)
@variable(model, x[C, W] >= 0)

@objective(model, Min, sum(t[c, w] * x[c, w] for c in C, w in W))
@constraint(model, supply[c in C], sum(x[c, w] for w in W) <= s[c])
@constraint(model, demand[w in W], sum(x[c, w] for c in C) >= d[w])

optimize!(model)
objective_value(model)
value.(x)


Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: Apple Accelerate 
LP has 5 rows; 6 cols; 12 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [1e+00, 2e+00]
  Bound   [0e+00, 0e+00]
  RHS     [3e+02, 6e+02]
Presolving model
5 rows, 6 cols, 12 nonzeros 0s
5 rows, 6 cols, 12 nonzeros 0s
Presolve reductions: rows 5(-0); columns 6(-0); nonzeros 12(-0) - Not reduced
Problem not reduced by presolve: solving the LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Pr: 3(900) 0.0s
          4     1.6800000000e+03 Pr: 0(0) 0.0s

Model status        : Optimal
Simplex   iterations: 4
Objective value     :  1.6800000000e+03
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, ["Seattle", "San Diego"]
    Dimension 2, ["New York", "Chicago", "Kansas City"]
And data, a 2×3 Matrix{Float64}:
  50.0  300.0    0.0
 250.0    0.0  300.0